In [ ]:
# Electricity Consumption Analysis SQL Queries
# Database: Bareilly_2021

import pandas as pd
import numpy as np

# 1. Load dataset
file_path = r"data/Bareilly_2021.csv"

df = pd.read_csv(file_path, encoding='latin1', low_memory=False)

df.columns = df.columns.str.strip()

df.rename(columns={
    'x_Timestamp': 'datetime',
    't_kWh': 'consumption',
    'meter': 'user_id'
}, inplace=True)

# 2. Cleaning missing values

# Convert datetime
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')

# Convert consumption
df['consumption'] = pd.to_numeric(df['consumption'], errors='coerce')

# Drop invalid rows
df = df.dropna(subset=['datetime', 'user_id', 'consumption'])

df['hour'] = df['datetime'].dt.hour
df['month'] = df['datetime'].dt.month
df['day'] = df['datetime'].dt.day

# Sort data by datetime
df = df.sort_values('datetime')


# 3. Analyze usage per user
user_usage = df.groupby('user_id')['consumption'].sum().sort_values(ascending=False)
print("\nTop Users:\n", user_usage.head())


# 4. Calculate moving average (20-hour window)
df['moving_avg'] = df['consumption'].rolling(window=20, min_periods=1).mean()


# 5. Segment users by usage level
def segment(x):
    if x < 0.2:
        return "Low"
    elif x < 1:
        return "Medium"
    else:
        return "High"

df['segment'] = df['consumption'].apply(segment)


# 6. Peak demand (hour-wise)
peak_demand = df.groupby('hour')['consumption'].sum().sort_values(ascending=False)
print("\nPeak Demand Hours:\n", peak_demand.head())

# 7. High consumption users
high_users = user_usage.head(10)
print("\nTop 10 High Users:\n", high_users)


# Monthly consumption
monthly_usage = df.groupby('month')['consumption'].sum()
print("\nMonthly Usage:\n", monthly_usage)


# Save cleaned data
df.to_csv("data/cleaned_bareilly_data.csv", index=False)

print("\nAll tasks completed successfully!")

